In [ ]:
# Cell 1: Install required libraries
%pip install -q imbalanced-learn shap lime

# Import basic libraries
import warnings
warnings.filterwarnings("ignore")

import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn preprocessing and model evaluation
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    PrecisionRecallDisplay
)

# Five machine learning models
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

# Imbalanced learning
import imblearn
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler

# Explainable AI
import shap
import lime
from lime.lime_tabular import LimeTabularExplainer

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Python version:", sys.version.split()[0])
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)
print("Scikit-learn version:", sklearn.__version__)
print("Imbalanced-learn version:", imblearn.__version__)
print("SHAP version:", shap.__version__)
print("Libraries imported successfully.")

In [ ]:
# Cell 2: Load the dataset

from google.colab import files
import os

FILE_PATH = "/content/drive/MyDrive/train_transactions.csv"

if not os.path.exists(FILE_PATH):
    print("Please upload train_transactions.csv")
    uploaded = files.upload()

    if "train_transactions.csv" in uploaded:
        FILE_PATH = "/content/train_transactions.csv"
    elif len(uploaded) == 1:
        FILE_PATH = "/content/" + next(iter(uploaded))
    else:
        raise FileNotFoundError(
            "Upload exactly one CSV file named train_transactions.csv."
        )

df = pd.read_csv("/content/drive/MyDrive/train_transactions.csv")

if df.empty:
    raise ValueError("The uploaded dataset contains no rows.")

print("Dataset loaded successfully.")
print("Dataset shape:", df.shape)

print("\nFirst five rows:")
display(df.head())

print("\nLast five rows:")
display(df.tail())

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nDescriptive statistics:")
display(df.describe(include="all").T)

print("\nMemory usage:")
print(f"{df.memory_usage(deep=True).sum() / 1024:.2f} KB")

In [ ]:
# --- User History Feature Engineering Cell ---
# Group by customer to calculate historical behavior profiles
if 'customer_id' not in df.columns:
    # If customer_id doesn't exist in your raw CSV, let's create a synthetic one
    # based on location or index for demonstration purposes
    np.random.seed(42)
    df['customer_id'] = df['customer_location'].apply(lambda x: f"cust_{abs(hash(x)) % 100}")

# Compute user-centric aggregation features
df['user_avg_amount'] = df.groupby('customer_id')['amount'].transform('mean')
df['user_txn_count'] = df.groupby('customer_id')['amount'].transform('count')

print("User history features successfully engineered!")
print(df[['customer_id', 'amount', 'user_avg_amount', 'user_txn_count']].head())

In [ ]:
# Cell 3: Initial data analysis

print("Missing values per column:")
display(df.isnull().sum().to_frame("Missing Values"))

print("\nUnique values per column:")
display(df.nunique(dropna=False).to_frame("Unique Values"))

print("\nNumerical columns:")
numerical_cols = df.select_dtypes(include=np.number).columns.tolist()
print(numerical_cols)

print("\nCategorical columns:")
categorical_cols = df.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()
print(categorical_cols)

# Confirm target column
TARGET = "is_fraud"

if TARGET not in df.columns:
    raise ValueError(
        f"Target column '{TARGET}' was not found. "
        "Inspect df.columns and set TARGET to the correct fraud-label column."
    )

# Validate binary target
if df[TARGET].isna().any():
    raise ValueError("The target column contains missing values.")

if set(df[TARGET].unique()) != {0, 1}:
    raise ValueError(
        "Expected is_fraud to contain exactly 0 (legitimate) and 1 (fraud). "
        "Inspect and correctly encode the target labels before continuing."
    )

print("\nTarget class distribution:")
class_counts = df[TARGET].value_counts().sort_index()
display(class_counts.rename(index={0: "Legitimate", 1: "Fraud"}))

print("\nTarget class percentages:")
display(
    (df[TARGET].value_counts(normalize=True) * 100)
    .sort_index()
    .rename(index={0: "Legitimate", 1: "Fraud"})
    .round(2)
    .to_frame("Percentage")
)

# Visualize imbalance
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x=TARGET)
plt.title("Legitimate vs Fraudulent Transactions")
plt.xlabel("is_fraud (0 = Legitimate, 1 = Fraud)")
plt.ylabel("Number of Transactions")
plt.tight_layout()
plt.show()

# Identify possible identifier columns
possible_id_cols = [
    col for col in df.columns
    if "id" in col.lower() or "uuid" in col.lower()
]

print("Possible identifier columns:", possible_id_cols)

# Check whether the dataset contains repeated rows
print("Exact duplicate rows:", df.duplicated().sum())

# Check target leakage risk
print(
    "\nLeakage check: transaction_id will be excluded from features. "
    "Only columns known at prediction time should be used."
)

In [ ]:
# Cell 4: Feature engineering and initial data preparation

df = df.copy()

# Remove rows with missing target labels, if any.
# The earlier cell checks this and stops if labels are missing.
df = df.dropna(subset=[TARGET]).copy()

# Convert timestamp to datetime
if "timestamp" in df.columns:
    df["timestamp"] = pd.to_datetime(
        df["timestamp"], errors="coerce"
    )

    # Extract time-related features
    df["transaction_hour"] = df["timestamp"].dt.hour
    df["transaction_day"] = df["timestamp"].dt.day
    df["transaction_month"] = df["timestamp"].dt.month
    df["transaction_weekday"] = df["timestamp"].dt.dayofweek

    # Drop raw timestamp after feature extraction
    df = df.drop(columns=["timestamp"])

    print("Time features created successfully.")
else:
    print("No timestamp column found; skipping time feature extraction.")

# Remove identifier columns from the feature set
identifier_cols = [
    col for col in df.columns
    if col.lower() in ["transaction_id", "id", "uuid"]
    or col.lower().endswith("_id")
]

identifier_cols = [col for col in identifier_cols if col != TARGET]

print("Identifier columns excluded from model features:", identifier_cols)

print("\nPrepared dataset shape:", df.shape)
print("\nPrepared columns:")
print(df.columns.tolist())

print("\nPrepared dataset preview:")
display(df.head())

print("\nData types after feature engineering:")
print(df.dtypes)

In [ ]:
# Cell 5: Outlier detection using IQR

numeric_features = df.drop(columns=[TARGET]).select_dtypes(
    include=np.number
).columns.tolist()

if not numeric_features:
    print("No numerical features available for outlier detection.")
else:
    outlier_summary = []

    for col in numeric_features:
        series = df[col].dropna()

        if series.empty:
            continue

        Q1 = series.quantile(0.25)
        Q3 = series.quantile(0.75)
        IQR = Q3 - Q1

        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        outlier_mask = (
            (df[col] < lower_bound) |
            (df[col] > upper_bound)
        )

        outlier_summary.append({
            "Feature": col,
            "Q1": Q1,
            "Q3": Q3,
            "IQR": IQR,
            "Lower Bound": lower_bound,
            "Upper Bound": upper_bound,
            "Outlier Count": int(outlier_mask.sum())
        })

    outlier_df = pd.DataFrame(outlier_summary)
    display(outlier_df)

    # Boxplots for numerical features
    for col in numeric_features:
        plt.figure(figsize=(7, 3))
        sns.boxplot(x=df[col])
        plt.title(f"Outlier Inspection: {col}")
        plt.xlabel(col)
        plt.tight_layout()
        plt.show()

print(
    "Note: IQR flags unusual values, not necessarily errors. "
    "Potentially fraudulent transactions must not be deleted merely "
    "because their amounts are unusual."
)

In [ ]:
# Cell 6: Duplicate removal

rows_before = len(df)
duplicates_before = int(df.duplicated().sum())

print("Rows before duplicate removal:", rows_before)
print("Exact duplicate rows found:", duplicates_before)

# Remove only exact duplicate rows
df = df.drop_duplicates().reset_index(drop=True)

rows_after = len(df)
duplicates_removed = rows_before - rows_after

print("\nExact duplicate rows removed:", duplicates_removed)
print("Rows after duplicate removal:", rows_after)
print("Updated dataset shape:", df.shape)
print("Remaining exact duplicates:", int(df.duplicated().sum()))

print("\nClass distribution after duplicate removal:")
display(df[TARGET].value_counts().sort_index().rename(
    index={0: "Legitimate", 1: "Fraud"}
))

In [ ]:
# Cell 7: Split into training and testing sets

X = df.drop(columns=[TARGET]).copy()
y = df[TARGET].astype(int).copy()

# Remove identifier columns
X = X.drop(
    columns=[col for col in identifier_cols if col in X.columns],
    errors="ignore"
)

# Check that the feature set is not empty
if X.shape[1] == 0:
    raise ValueError("No features remain after excluding identifiers.")

# Check whether stratification is possible
class_counts = y.value_counts()

if class_counts.min() < 2:
    raise ValueError(
        "At least two examples of each class are needed for a stratified split."
    )

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training feature shape:", X_train.shape)
print("Testing feature shape:", X_test.shape)

print("\nTraining class distribution:")
display(y_train.value_counts().sort_index().rename(
    index={0: "Legitimate", 1: "Fraud"}
))

print("\nTesting class distribution:")
display(y_test.value_counts().sort_index().rename(
    index={0: "Legitimate", 1: "Fraud"}
))

print("\nTraining fraud percentage:",
      round(100 * y_train.mean(), 2), "%")

print("Testing fraud percentage:",
      round(100 * y_test.mean(), 2), "%")

print(
    "\nThe test set is not resampled. It retains the original "
    "class proportions as closely as stratification permits."
)

In [ ]:
# Cell 8: Preprocessing pipeline

numeric_features = X_train.select_dtypes(
    include=np.number
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("Numerical features:", numeric_features)
print("Categorical features:", categorical_features)

# Numerical preprocessing
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

# Combine preprocessing
transformers = []

if numeric_features:
    transformers.append(
        ("num", numeric_transformer, numeric_features)
    )

if categorical_features:
    transformers.append(
        ("cat", categorical_transformer, categorical_features)
    )

preprocessor = ColumnTransformer(
    transformers=transformers,
    remainder="drop",
    verbose_feature_names_out=False
)

# Inspect transformed feature names
preprocessor.fit(X_train)

feature_names = preprocessor.get_feature_names_out()

print("\nNumber of transformed features:", len(feature_names))
print("First 20 transformed feature names:")
print(feature_names[:20])

# Show original class imbalance before resampling
print("\nTraining class distribution before resampling:")
display(y_train.value_counts().sort_index())

# The sampler is placed inside each model's pipeline.
# It will run during fit() on training data only.
print(
    "\nPreprocessing is ready. RandomOverSampler will be applied "
    "inside each training pipeline, not to the test set."
)

In [ ]:
# Cell 9: Define and train five ML models

models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),

    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ),

    "SVM": SVC(
        kernel="rbf",
        probability=True,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ),

    "KNN": KNeighborsClassifier(
        n_neighbors=5,
        weights="distance"
    ),

    "Decision Tree": DecisionTreeClassifier(
        class_weight="balanced",
        random_state=RANDOM_STATE
    )
}

trained_models = {}

for model_name, estimator in models.items():
    print(f"\nTraining {model_name}...")

    # Each model receives its own unfitted preprocessing pipeline.
    model_pipeline = ImbPipeline(steps=[
        ("preprocessor", preprocessor),
        ("sampler", RandomOverSampler(
            random_state=RANDOM_STATE
        )),
        ("classifier", estimator)
    ])

    model_pipeline.fit(X_train, y_train)

    trained_models[model_name] = model_pipeline

    print(f"{model_name} trained successfully.")

print("\nAll five models have been trained.")

In [ ]:
# Cell 10: Evaluate all five models

results = []
predictions = {}
probabilities = {}

for model_name, model_pipeline in trained_models.items():

    # Predict labels on untouched test data
    y_pred = model_pipeline.predict(X_test)
    predictions[model_name] = y_pred

    # Get fraud probability, if the model supports it
    if hasattr(model_pipeline, "predict_proba"):
        y_prob = model_pipeline.predict_proba(X_test)[:, 1]
        probabilities[model_name] = y_prob
    else:
        y_prob = None

    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(
        y_test, y_pred, pos_label=1, zero_division=0
    )
    recall = recall_score(
        y_test, y_pred, pos_label=1, zero_division=0
    )
    f1 = f1_score(
        y_test, y_pred, pos_label=1, zero_division=0
    )

    # AUC metrics require both classes in the test set
    roc_auc = (
        roc_auc_score(y_test, y_prob)
        if y_prob is not None and y_test.nunique() == 2
        else np.nan
    )

    pr_auc = (
        average_precision_score(y_test, y_prob)
        if y_prob is not None
        else np.nan
    )

    results.append({
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-Score": f1,
        "ROC-AUC": roc_auc,
        "PR-AUC": pr_auc
    })

    print("\n" + "=" * 60)
    print("MODEL:", model_name)
    print("=" * 60)

    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print(f"ROC-AUC:   {roc_auc:.4f}" if not np.isnan(roc_auc)
          else "ROC-AUC: Not available")
    print(f"PR-AUC:    {pr_auc:.4f}" if not np.isnan(pr_auc)
          else "PR-AUC: Not available")

    print("\nClassification report:")
    print(classification_report(
        y_test,
        y_pred,
        labels=[0, 1],
        target_names=["Legitimate", "Fraud"],
        zero_division=0
    ))

    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])

    ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["Legitimate", "Fraud"]
    ).plot(cmap="Blues", values_format="d")

    plt.title(f"Confusion Matrix - {model_name}")
    plt.tight_layout()
    plt.show()

print("\nEvaluation of all models completed.")

In [ ]:
# ============================================================
# CELL 11 (CORRECTED): Model Comparison & Visualizations
# ============================================================

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(
    by="F1-Score", ascending=False
).reset_index(drop=True)

print("Model comparison (sorted by fraud-class F1-score):")
display(
    results_df.style.format({
        "Accuracy": "{:.4f}",
        "Precision": "{:.4f}",
        "Recall": "{:.4f}",
        "F1-Score": "{:.4f}",
        "ROC-AUC": "{:.4f}",
        "PR-AUC": "{:.4f}",
    })
)

# 1. Visual comparison of key metrics
metrics_to_plot = ["Accuracy", "Precision", "Recall", "F1-Score", "PR-AUC"]
results_plot = results_df.set_index("Model")[metrics_to_plot]

fig, ax = plt.subplots(figsize=(12, 6))
results_plot.plot(kind="bar", ylim=(0, 1), ax=ax)
plt.title("Comparison of Five Fraud Detection Models")
plt.ylabel("Score")
plt.xlabel("Machine Learning Model")
plt.xticks(rotation=30, ha="right")
plt.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=3)
plt.tight_layout()
plt.show()

# Select the best model by F1-score
best_model_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]

print("Selection criterion: Highest fraud-class F1-score")
print("Selected best model:", best_model_name)

print("\nDetailed report for the selected model:")
print(
    classification_report(
        y_test,
        predictions[best_model_name],
        labels=[0, 1],
        target_names=["Legitimate", "Fraud"],
        zero_division=0,
    )
)

plt.figure(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    predictions[best_model_name],
    labels=[0, 1],
    display_labels=["Legitimate", "Fraud"],
    cmap="Blues",
    values_format="d",
)
plt.title(f"Best Model: {best_model_name}")
plt.tight_layout()
plt.show()

# 2. ROC Curve Comparison with clean legend placement
if y_test.nunique() == 2:
  plt.figure(figsize=(8, 6))
  ax_roc = plt.gca()
  for model_name, y_prob in probabilities.items():
    RocCurveDisplay.from_predictions(
        y_test, y_prob, name=model_name, ax=ax_roc
    )
  plt.title("ROC Curve Comparison")
  plt.legend(loc="lower right")
  plt.tight_layout()
  plt.show()

  # 3. Precision-Recall Curve Comparison with clean legend placement
  plt.figure(figsize=(8, 6))
  ax_pr = plt.gca()
  for model_name, y_prob in probabilities.items():
    PrecisionRecallDisplay.from_predictions(
        y_test, y_prob, name=model_name, ax=ax_pr
    )
  plt.title("Precision-Recall Curve Comparison")
  plt.legend(loc="upper right")
  plt.tight_layout()
  plt.show()

In [ ]:
# Cell 12: Explainable AI using SHAP

# Transform features using the selected model's fitted preprocessor
fitted_preprocessor = best_model.named_steps["preprocessor"]
best_classifier = best_model.named_steps["classifier"]

X_test_transformed = fitted_preprocessor.transform(X_test)
X_train_transformed = fitted_preprocessor.transform(X_train)

# Convert to dense arrays for SHAP/LIME compatibility
X_test_transformed = np.asarray(X_test_transformed, dtype=float)
X_train_transformed = np.asarray(X_train_transformed, dtype=float)

# Use feature names from the fitted preprocessing pipeline
shap_feature_names = list(
    fitted_preprocessor.get_feature_names_out()
)

print("Selected model:", best_model_name)
print("Number of model input features:", len(shap_feature_names))
print("Test rows available:", len(X_test_transformed))

# Limit sample size for faster explanation
rng = np.random.default_rng(RANDOM_STATE)

n_background = min(30, len(X_train_transformed))
n_explain = min(20, len(X_test_transformed))

background_indices = rng.choice(
    len(X_train_transformed),
    size=n_background,
    replace=False
)

explain_indices = rng.choice(
    len(X_test_transformed),
    size=n_explain,
    replace=False
)

background_data = X_train_transformed[background_indices]
explain_data = X_test_transformed[explain_indices]

shap_values = None

try:
    # TreeExplainer is suitable for tree-based classifiers
    if best_model_name in ["Random Forest", "Decision Tree"]:
        shap_explainer = shap.TreeExplainer(best_classifier)
        shap_values = shap_explainer.shap_values(explain_data)

        # SHAP versions may return different formats for binary classes
        if isinstance(shap_values, list):
            shap_values = shap_values[1]
        elif isinstance(shap_values, np.ndarray) and shap_values.ndim == 3:
            shap_values = shap_values[:, :, 1]

    else:
        # Model-agnostic explanation for non-tree models
        shap_predict_function = best_classifier.predict_proba

        shap_explainer = shap.Explainer(
            shap_predict_function,
            background_data,
            feature_names=shap_feature_names
        )

        shap_result = shap_explainer(
            explain_data,
            max_evals=2 * len(shap_feature_names) + 1
        )

        # Extract the fraud-class SHAP values
        shap_values = shap_result.values

        if shap_values.ndim == 3:
            shap_values = shap_values[:, :, 1]

    print("SHAP values calculated successfully.")

    # Summary plot
    shap.summary_plot(
        shap_values,
        explain_data,
        feature_names=shap_feature_names,
        show=False,
        max_display=15
    )

    plt.title("SHAP Summary: Fraud Prediction")
    plt.tight_layout()
    plt.show()

    # Mean absolute SHAP feature importance
    mean_abs_shap = np.abs(shap_values).mean(axis=0)

    shap_importance = pd.DataFrame({
        "Feature": shap_feature_names,
        "Mean Absolute SHAP": mean_abs_shap
    }).sort_values(
        "Mean Absolute SHAP", ascending=False
    )

    print("\nTop 15 features by mean absolute SHAP value:")
    display(shap_importance.head(15))

    shap_importance.head(15).sort_values(
        "Mean Absolute SHAP"
    ).plot(
        x="Feature",
        y="Mean Absolute SHAP",
        kind="barh",
        figsize=(9, 6),
        legend=False
    )

    plt.title("Top Features by SHAP Importance")
    plt.xlabel("Mean Absolute SHAP Value")
    plt.ylabel("Feature")
    plt.tight_layout()
    plt.show()

except Exception as e:
    print("SHAP explanation could not be completed:", str(e))
    print(
        "Check the SHAP version, model compatibility, and transformed "
        "feature dimensions before interpreting the results."
    )

In [ ]:
# Cell 13: Explainable AI using LIME

# Reuse the fitted model's classifier and transformed data
lime_training_data = X_train_transformed
lime_test_data = X_test_transformed

lime_explainer = LimeTabularExplainer(
    training_data=lime_training_data,
    feature_names=shap_feature_names,
    class_names=["Legitimate", "Fraud"],
    mode="classification",
    discretize_continuous=True,
    random_state=RANDOM_STATE
)

# Choose one test transaction
normal_index = 0

# Choose a test transaction predicted as fraud, if available
fraud_indices = np.where(
    predictions[best_model_name] == 1
)[0]

if len(fraud_indices) > 0:
    fraud_index = int(fraud_indices[0])
else:
    fraud_index = None
    print(
        "The selected model did not predict any test transactions as fraud. "
        "Only a regular test transaction will be explained."
    )

def explain_transaction_lime(index, explanation_title):
    """Explain one transformed transaction using LIME."""

    row = lime_test_data[index]

    predicted_label = int(
        best_classifier.predict(row.reshape(1, -1))[0]
    )

    predicted_probabilities = best_classifier.predict_proba(
        row.reshape(1, -1)
    )[0]

    print("\n" + "=" * 60)
    print(explanation_title)
    print("=" * 60)

    print("Test row index:", index)
    print("Actual label:", int(y_test.iloc[index]))
    print("Predicted label:", predicted_label)
    print(
        "Prediction:",
        "Fraud" if predicted_label == 1 else "Legitimate"
    )
    print(
        f"Predicted fraud probability: "
        f"{predicted_probabilities[1]:.4f}"
    )

    explanation = lime_explainer.explain_instance(
        data_row=row,
        predict_fn=best_classifier.predict_proba,
        labels=(1,),
        num_features=min(10, len(shap_feature_names)),
        num_samples=2000
    )

    print("\nFeatures supporting or opposing the fraud prediction:")
    display(pd.DataFrame(
        explanation.as_list(label=1),
        columns=["Feature Condition", "LIME Weight"]
    ))

    fig = explanation.as_pyplot_figure(label=1)
    fig.set_size_inches(9, 5)
    plt.title(explanation_title)
    plt.tight_layout()
    plt.show()

# Explain one test transaction
explain_transaction_lime(
    normal_index,
    "LIME Explanation: Selected Test Transaction"
)

# Explain a predicted fraudulent transaction if available
if fraud_index is not None and fraud_index != normal_index:
    explain_transaction_lime(
        fraud_index,
        "LIME Explanation: Model-Predicted Fraud Transaction"
    )
elif fraud_index == normal_index:
    print(
        "\nThe first test transaction is already predicted as fraud; "
        "the explanation above covers that transaction."
    )

In [ ]:

# ============================================================
# CELL 14: COMPLETE FINAL MODEL EVALUATION
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    matthews_corrcoef,
    balanced_accuracy_score
)

# ------------------------------------------------------------
# 1. Calculate all evaluation measures for five ML models
# ------------------------------------------------------------

final_results = []

for model_name, model in trained_models.items():

    # Predictions on the untouched test set
    y_pred = model.predict(X_test)

    # Predicted probabilities for fraud class
    y_prob = None
    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)[:, 1]

    # Confusion matrix: 0 = legitimate, 1 = fraud
    tn, fp, fn, tp = confusion_matrix(
        y_test, y_pred, labels=[0, 1]
    ).ravel()

    # Performance metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(
        y_test, y_pred, zero_division=0
    )
    recall = recall_score(
        y_test, y_pred, zero_division=0
    )
    f1 = f1_score(
        y_test, y_pred, zero_division=0
    )
    balanced_acc = balanced_accuracy_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)

    roc_auc = (
        roc_auc_score(y_test, y_prob)
        if y_prob is not None and y_test.nunique() == 2
        else np.nan
    )

    pr_auc = (
        average_precision_score(y_test, y_prob)
        if y_prob is not None
        else np.nan
    )

    # Additional error rates
    specificity = tn / (tn + fp) if (tn + fp) else 0
    fpr = fp / (fp + tn) if (fp + tn) else 0
    fnr = fn / (fn + tp) if (fn + tp) else 0

    final_results.append({
        "Model": model_name,
        "Accuracy (%)": accuracy * 100,
        "Precision (%)": precision * 100,
        "Fraud Recall (%)": recall * 100,
        "F1-Score (%)": f1 * 100,
        "Specificity (%)": specificity * 100,
        "Balanced Accuracy (%)": balanced_acc * 100,
        "ROC-AUC (%)": roc_auc * 100 if not np.isnan(roc_auc) else np.nan,
        "PR-AUC (%)": pr_auc * 100 if not np.isnan(pr_auc) else np.nan,
        "MCC": mcc,
        "False Positive Rate (%)": fpr * 100,
        "False Negative Rate (%)": fnr * 100,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    })

# Create final table
final_df = pd.DataFrame(final_results)

print("\n" + "=" * 80)
print("FINAL PERFORMANCE COMPARISON OF FIVE ML MODELS")
print("=" * 80)

display(final_df.round(2))

# ------------------------------------------------------------
# 2. Identify the best model using fraud F1-score
# ------------------------------------------------------------

best_index = final_df["F1-Score (%)"].idxmax()
best_model_name = final_df.loc[best_index, "Model"]

print("\nSelected best model:", best_model_name)
print("Selection criterion: Highest fraud-class F1-score")

# ------------------------------------------------------------
# 3. Print classification report for every model
# ------------------------------------------------------------

for model_name, model in trained_models.items():

    y_pred = model.predict(X_test)

    print("\n" + "=" * 70)
    print("CLASSIFICATION REPORT:", model_name)
    print("=" * 70)

    print(classification_report(
        y_test,
        y_pred,
        labels=[0, 1],
        target_names=["Legitimate", "Fraud"],
        digits=4,
        zero_division=0
    ))

# ------------------------------------------------------------
# 4. Plot confusion matrices for all five models
# ------------------------------------------------------------

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, (model_name, model) in enumerate(trained_models.items()):

    y_pred = model.predict(X_test)

    cm = confusion_matrix(
        y_test, y_pred, labels=[0, 1]
    )

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["Legitimate", "Fraud"],
        yticklabels=["Legitimate", "Fraud"],
        ax=axes[i]
    )

    axes[i].set_title(model_name)
    axes[i].set_xlabel("Predicted Class")
    axes[i].set_ylabel("Actual Class")

# Hide unused sixth plot
axes[-1].axis("off")

plt.suptitle("Confusion Matrix Comparison", fontsize=16)
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 5. Accuracy percentage comparison
# ------------------------------------------------------------

accuracy_chart = final_df.sort_values(
    "Accuracy (%)", ascending=False
)

plt.figure(figsize=(10, 5))

bars = plt.bar(
    accuracy_chart["Model"],
    accuracy_chart["Accuracy (%)"]
)

plt.ylabel("Accuracy (%)")
plt.xlabel("Machine Learning Model")
plt.title("Accuracy Percentage of Five ML Models")
plt.xticks(rotation=25, ha="right")

for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        height,
        f"{height:.2f}%",
        ha="center",
        va="bottom"
    )

plt.ylim(0, 105)
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 6. Compare precision, recall, F1-score, and specificity
# ------------------------------------------------------------

comparison_metrics = [
    "Precision (%)",
    "Fraud Recall (%)",
    "F1-Score (%)",
    "Specificity (%)"
]

final_df.set_index("Model")[comparison_metrics].plot(
    kind="bar",
    figsize=(12, 6)
)

plt.title("Comparison of Important Fraud Detection Metrics")
plt.ylabel("Percentage (%)")
plt.xlabel("Machine Learning Model")
plt.xticks(rotation=25, ha="right")
plt.ylim(0, 105)
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 7. Detailed confusion matrix for the best model
# ------------------------------------------------------------

best_model = trained_models[best_model_name]
best_predictions = best_model.predict(X_test)

tn, fp, fn, tp = confusion_matrix(
    y_test, best_predictions, labels=[0, 1]
).ravel()

print("\n" + "=" * 70)
print("BEST MODEL: DETAILED CONFUSION MATRIX")
print("=" * 70)

print(f"Model: {best_model_name}")
print(f"True Negatives (TN): {tn}")
print(f"False Positives (FP): {fp}")
print(f"False Negatives (FN): {fn}")
print(f"True Positives (TP): {tp}")

print("\nBest model accuracy:",
      f"{accuracy_score(y_test, best_predictions) * 100:.2f}%")

print("\nBest model classification report:")
print(classification_report(
    y_test,
    best_predictions,
    labels=[0, 1],
    target_names=["Legitimate", "Fraud"],
    digits=4,
    zero_division=0
))

# ------------------------------------------------------------
# 8. Export results for the TRW report
# ------------------------------------------------------------

final_df.to_csv(
    "fraud_detection_final_results.csv",
    index=False
)

print("\nFinal evaluation completed.")
print("Results saved to fraud_detection_final_results.csv")


In [ ]:
# Select the already-trained Random Forest model
rf_model = trained_models["Random Forest"]

# Get the exact features used during training
feature_columns = list(X_train.columns)

print("Random Forest model selected successfully!")
print("Input features:", feature_columns)
print("Model classes:", rf_model.classes_)

In [ ]:
!pip install -q gradio

In [ ]:

import gradio as gr
import pandas as pd
import numpy as np
import html

# ==========================================
# 1. LOAD EXISTING TRAINED MODEL
# ==========================================

rf_model = trained_models["Random Forest"]

feature_columns = list(X_train.columns)

location_options = sorted(
    X_train["customer_location"].dropna().astype(str).unique().tolist()
)

payment_options = sorted(
    X_train["payment_method"].dropna().astype(str).unique().tolist()
)


# ==========================================
# 2. DARK ROYAL BLUE THEME
# ==========================================

custom_css = """

@import url('https://fonts.googleapis.com/css2?family=Poppins:wght@400;500;600;700;800&family=Inter:wght@400;500;600&display=swap');

/* REMOVE BLACK BACKGROUND */

html, body {
    background: #123B75 !important;
    margin: 0 !important;
    padding: 0 !important;
    font-family: 'Inter', sans-serif !important;
}

/* FULL SCREEN WIDTH */

.gradio-container {
    background: linear-gradient(
        135deg,
        #123B75,
        #1B4F91,
        #123B75
    ) !important;

    color: #FFFFFF !important;

    width: 100% !important;
    max-width: 100% !important;
    min-height: 100vh !important;

    margin: 0 !important;
    padding: 18px 28px !important;

    box-sizing: border-box !important;

    position: relative !important;
    isolation: isolate;
}

/* REMOVE CENTRED CONTENT LIMIT */

.gradio-container .main,
.gradio-container .wrap {
    max-width: 100% !important;
    width: 100% !important;
}

/* FULL PAGE WATERMARK */

.gradio-container::before {
    content: "FRAUD DETECTION";

    position: fixed;
    top: 42%;
    left: 0;
    width: 100%;

    text-align: center;

    font-family: 'Poppins', sans-serif;
    font-size: clamp(35px, 8vw, 110px);
    font-weight: 800;

    letter-spacing: 8px;

    color: rgba(255,255,255,0.045);

    transform: rotate(-20deg);

    pointer-events: none;
    z-index: -1;

    white-space: nowrap;
}

/* HEADINGS */

h1, h2, h3 {
    font-family: 'Poppins', sans-serif !important;
    color: #FFFFFF !important;
    font-weight: 700 !important;
}

/* GENERAL TEXT */

p, span {
    font-family: 'Inter', sans-serif;
    color: #EAF2FF;
}

/* INPUT LABELS */

label {
    color: #FFFFFF !important;
    font-weight: 600 !important;
    font-size: 15px !important;
}

/* INPUT FIELDS */

input, textarea {
    background: #244F82 !important;
    color: #FFFFFF !important;

    border: 1px solid #75A9E8 !important;
    border-radius: 9px !important;
}

/* PLACEHOLDER */

input::placeholder {
    color: #C6D9F2 !important;
}

/* DROPDOWN */

.wrap-inner, .secondary-wrap {
    background: #244F82 !important;
    border-radius: 9px !important;
    color: white !important;
}

/* PANELS */

.block {
    background: rgba(27, 79, 145, 0.85) !important;
    border: 1px solid #477DBB !important;
    border-radius: 12px !important;
}

/* DETECT BUTTON */

button.primary {
    background: #75A9E8 !important;
    color: #102E55 !important;

    font-family: 'Poppins', sans-serif !important;
    font-size: 17px !important;
    font-weight: 800 !important;

    border: none !important;
    border-radius: 10px !important;

    min-height: 52px !important;
}

button.primary:hover {
    background: #FFFFFF !important;
    color: #123B75 !important;
}

/* RESULT CARD */

.result-card {
    background: linear-gradient(
        135deg,
        #1B4F91,
        #244F82
    );

    border: 1px solid #75A9E8;
    border-radius: 15px;

    padding: 24px;
    margin-top: 16px;

    color: #FFFFFF;
}

/* FRAUD TEXT - RED ONLY FOR FRAUD */

.fraud-text {
    color: #FF5555 !important;
    font-weight: 800;
}

/* LEGITIMATE TEXT - GREEN */

.legit-text {
    color: #65E69A !important;
    font-weight: 800;
}

/* PROBABILITY SECTION */

.prob-label {
    display: flex;
    justify-content: space-between;
    align-items: center;

    color: #FFFFFF;

    font-size: 17px;
    font-weight: 700;

    margin: 18px 0 9px;
}

/* PROBABILITY TRACK */

.prob-track {
    height: 16px;
    background: #5278A9;

    border-radius: 20px;
    overflow: hidden;
}

/* BLUE FRAUD PROBABILITY BAR */

.prob-fill {
    height: 100%;
    border-radius: 20px;
}

.fraud-bar {
    background: linear-gradient(
        90deg,
        #468FD8,
        #A5D1FF
    );
}

/* GREEN LEGITIMATE PROBABILITY BAR */

.legit-bar {
    background: linear-gradient(
        90deg,
        #20A969,
        #65E69A
    );
}

/* TRANSACTION DETAILS */

.details-card {
    background: #1B4F91;

    border: 1px solid #75A9E8;
    border-radius: 12px;

    padding: 22px;
    margin-top: 16px;

    color: #FFFFFF;
    line-height: 2;
}

.details-card p {
    color: #EAF2FF;
}

/* FOOTER */

.footer-text {
    text-align: center;
    color: #DCEBFF;

    font-size: 13px;
    letter-spacing: 2px;

    padding: 20px;
}

/* DIVIDER */

hr {
    border-color: #75A9E8 !important;
}

/* MOBILE RESPONSIVE */

@media (max-width: 700px) {

    .gradio-container {
        padding: 12px !important;
    }

    h1 {
        font-size: 23px !important;
    }

    .prob-label {
        font-size: 14px;
    }

}

"""


# ==========================================
# 3. FRAUD DETECTION FUNCTION
# ==========================================

def detect_fraud(amount, timestamp, location, payment_method):

    try:

        # VALIDATION

        if amount is None or float(amount) < 0:
            return "<h3>Please enter a valid amount.</h3>", ""

        if location is None or payment_method is None:
            return "<h3>Please select location and payment method.</h3>", ""

        transaction_time = pd.to_datetime(
            timestamp,
            errors="coerce"
        )

        if pd.isna(transaction_time):
            return "<h3>Please enter a valid timestamp.</h3>", ""

        # PREPARE INPUT FEATURES

        input_data = {
            "amount": float(amount),
            "customer_location": str(location),
            "payment_method": str(payment_method),
            "transaction_hour": transaction_time.hour,
            "transaction_day": transaction_time.day,
            "transaction_month": transaction_time.month,
            "transaction_weekday": transaction_time.dayofweek
        }

        input_df = pd.DataFrame(
            [input_data],
            columns=feature_columns
        )

        # MODEL PREDICTION

        prediction = int(rf_model.predict(input_df)[0])

        # PREDICTION PROBABILITIES

        probabilities = rf_model.predict_proba(input_df)[0]

        classes = list(rf_model.classes_)

        fraud_probability = float(
            probabilities[classes.index(1)] * 100
        )

        legitimate_probability = float(
            probabilities[classes.index(0)] * 100
        )

        # ==================================
        # PREDICTION RESULT
        # ==================================

        if prediction == 1:

            result_html = """

            <div class="result-card">

                <h1 class="fraud-text"
                    style="font-size:30px;">

                    🚨 FRAUDULENT TRANSACTION

                </h1>

                <h3 style="color:white !important;">
                    Suspicious Activity Detected
                </h3>

                <p>
                    The Random Forest model has classified
                    this transaction as potentially fraudulent.
                </p>

            </div>

            """

        else:

            result_html = """

            <div class="result-card">

                <h1 class="legit-text"
                    style="font-size:30px;">

                    ✅ LEGITIMATE TRANSACTION

                </h1>

                <h3 style="color:white !important;">
                    Transaction Appears Safe
                </h3>

                <p>
                    The Random Forest model has classified
                    this transaction as legitimate.
                </p>

            </div>

            """

        # ==================================
        # PROBABILITY DOUBLE LINE DISPLAY
        # ==================================

        fraud_width = max(0, min(100, fraud_probability))
        legit_width = max(0, min(100, legitimate_probability))

        result_html += f"""

        <div class="result-card">

            <h2>📊 Prediction Probability Analysis</h2>

            <div class="prob-label">

                <span class="fraud-text">
                    FRAUD PROBABILITY
                </span>

                <span>
                    {fraud_probability:.2f}%
                </span>

            </div>

            <div class="prob-track">

                <div class="prob-fill fraud-bar"
                     style="width:{fraud_width:.2f}%;">
                </div>

            </div>

            <div class="prob-label">

                <span class="legit-text">
                    LEGITIMATE PROBABILITY
                </span>

                <span>
                    {legitimate_probability:.2f}%
                </span>

            </div>

            <div class="prob-track">

                <div class="prob-fill legit-bar"
                     style="width:{legit_width:.2f}%;">
                </div>

            </div>

            <br>

            <p>
                <b>Prediction Algorithm:</b> Random Forest
            </p>

        </div>

        """

        # ==================================
        # TRANSACTION DETAILS
        # ==================================

        safe_location = html.escape(str(location))
        safe_payment = html.escape(str(payment_method))

        details_html = f"""

        <div class="details-card">

            <h2>💳 Transaction Information</h2>

            <p>
                <b>Transaction Amount:</b>
                ₹{float(amount):,.2f}
            </p>

            <p>
                <b>Timestamp:</b>
                {transaction_time}
            </p>

            <p>
                <b>Customer Location:</b>
                {safe_location}
            </p>

            <p>
                <b>Payment Method:</b>
                {safe_payment}
            </p>

            <hr>

            <h3>⚙️ Automatically Generated Features</h3>

            <p>
                <b>Transaction Hour:</b>
                {transaction_time.hour}
            </p>

            <p>
                <b>Transaction Day:</b>
                {transaction_time.day}
            </p>

            <p>
                <b>Transaction Month:</b>
                {transaction_time.month}
            </p>

            <p>
                <b>Transaction Weekday:</b>
                {transaction_time.dayofweek}
            </p>

        </div>

        """

        return result_html, details_html

    except Exception as e:

        return (
            f"<h3>Prediction Error: {html.escape(str(e))}</h3>",
            ""
        )


# ==========================================
# 4. CREATE FULL-WIDTH GRADIO INTERFACE
# ==========================================

with gr.Blocks(
    theme=gr.themes.Soft(
        primary_hue="blue",
        secondary_hue="blue",
        neutral_hue="slate"
    ),
    css=custom_css,
    title="Financial Transaction Fraud Detection"
) as demo:

    # HEADER

    gr.HTML("""

    <div style="
        text-align:center;
        padding:15px;
    ">

        <h1 style="
            font-size:34px;
            color:white !important;
        ">

            🔍 FINANCIAL TRANSACTION FRAUD DETECTION

        </h1>

        <h3 style="color:#DCEBFF !important;">

            Machine Learning Based Fraud Detection System

        </h3>

        <p style="color:#FFFFFF;">

            Enter transaction details to identify whether
            the financial transaction is fraudulent or legitimate.

        </p>

        <h3 style="color:#A5D1FF !important;">

            BANKING SECURITY • DIGITAL PAYMENTS • AI MONITORING

        </h3>

    </div>

    """)

    # ======================================
    # BANKING AND SECURITY IMAGES
    # ======================================

    gr.HTML("""

    <div style="
        display:grid;
        grid-template-columns:repeat(3,minmax(0,1fr));
        gap:18px;
        width:100%;
        margin:15px 0;
    ">

        <div style="text-align:center;">

            <img
            src="https://images.unsplash.com/photo-1556742049-0cfed4f6a45d?w=700"
            style="
                width:100%;
                height:190px;
                object-fit:cover;
                border-radius:12px;
                border:2px solid #75A9E8;
            "
            alt="Digital payment security">

            <h3>Digital Payments</h3>

        </div>

        <div style="text-align:center;">

            <img
            src="https://images.unsplash.com/photo-1563013544-824ae1b704d3?w=700"
            style="
                width:100%;
                height:190px;
                object-fit:cover;
                border-radius:12px;
                border:2px solid #75A9E8;
            "
            alt="Credit card banking">

            <h3>Credit Card Security</h3>

        </div>

        <div style="text-align:center;">

            <img
            src="https://images.unsplash.com/photo-1554224155-6726b3ff858f?w=700"
            style="
                width:100%;
                height:190px;
                object-fit:cover;
                border-radius:12px;
                border:2px solid #75A9E8;
            "
            alt="Financial analytics">

            <h3>Financial Protection</h3>

        </div>

    </div>

    """)

    gr.Markdown("---")

    # ======================================
    # INPUT SECTION
    # ======================================

    gr.Markdown("## 💳 ENTER TRANSACTION DETAILS")

    with gr.Row():

        amount_input = gr.Number(
            label="Transaction Amount (₹)",
            value=None,
            precision=2
        )

        timestamp_input = gr.Textbox(
            label="Transaction Timestamp",
            placeholder="YYYY-MM-DD HH:MM:SS",
            value="2026-11-13 10:30:00"
        )

    with gr.Row():

        location_input = gr.Dropdown(
            choices=location_options,
            label="Customer Location",
            value=location_options[0] if location_options else None
        )

        payment_input = gr.Dropdown(
            choices=payment_options,
            label="Payment Method",
            value=payment_options[0] if payment_options else None
        )

    # DETECT BUTTON

    predict_button = gr.Button(
        "🔍 DETECT FRAUD",
        variant="primary"
    )

    gr.Markdown("---")

    # ======================================
    # OUTPUT SECTION
    # ======================================

    gr.Markdown("## 📊 PREDICTION RESULT")

    result_output = gr.HTML()

    details_output = gr.HTML()

    predict_button.click(
        fn=detect_fraud,
        inputs=[
            amount_input,
            timestamp_input,
            location_input,
            payment_input
        ],
        outputs=[
            result_output,
            details_output
        ]
    )

    # ======================================
    # FOOTER
    # ======================================

    gr.HTML("""

    <div class="footer-text">

        <h3>🛡️ FRAUD DETECTION SYSTEM</h3>

        <p>
            Powered by Machine Learning and Financial Analytics
        </p>

        <p>
            Target: 0 = Legitimate | 1 = Fraud
        </p>

        <p>
            FINANCIAL TRANSACTION SECURITY
        </p>

    </div>

    """)


# ==========================================
# 5. LAUNCH APPLICATION
# ==========================================

demo.launch(share=True)
